###1. Create a SparkSession: Write the basic code to initialize a SparkSession with a specific app name. 

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataEngineeringJob") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

In [0]:
spark.stop()

###2. Read CSV with Header: Read a CSV file while inferring the schema and specifying that the first row is a header.

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/dbx/scenarios/files/annual-enterprise-survey.csv")

df.show()
df.printSchema()

###3. Select & Rename: Select three columns from a DataFrame and rename one of them using alias().

In [0]:
# Sample DataFrame
data = [
    (1, "Bharath", 28, "India"),
    (2, "Ravi", 30, "USA")
]

columns = ["id", "name", "age", "country"]

df = spark.createDataFrame(data, columns)

# Select three columns and rename one using alias()
df_selected = df.select(
    "id",
    df.name.alias("full_name"),   # Renamed column
    "age"
)

df_selected.show()

###4. Filter with Multiple Conditions: Filter rows where salary > 5000 and department == 'Sales'

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, "Bharath", 6000, "Sales"),
    (2, "Ravi", 4500, "Sales"),
    (3, "Anu", 7000, "HR"),
    (4, "John", 8000, "Sales")
]

columns = ["id", "name", "salary", "department"]

df = spark.createDataFrame(data, columns)

# Filter where salary > 5000 AND department == 'Sales'
filtered_df = df.filter(
    (col("salary") > 5000) & (col("department") == "Sales")
)

filtered_df.show()

###5. Add a New Column: Create a new column total_price by multiplying quantity and unit_price

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, 2, 500),
    (2, 5, 200),
    (3, 3, 1000)
]

columns = ["product_id", "quantity", "unit_price"]

df = spark.createDataFrame(data, columns)

# Add new column total_price
df_new = df.withColumn(
    "total_price",
    col("quantity") * col("unit_price")
)

df_new.show()

###6. Remove Duplicates: Use dropDuplicates() to remove rows based on specific primary key columns. 

In [0]:
# Sample DataFrame
data = [
    (1, "Bharath", "Sales"),
    (2, "Ravi", "HR"),
    (1, "Bharath", "Sales"),   # Duplicate row
    (3, "Anu", "IT"),
    (2, "Ravi", "HR")          # Duplicate row
]

columns = ["emp_id", "name", "department"]

df = spark.createDataFrame(data, columns)

# Remove duplicates based on primary key column (emp_id)
df_dedup = df.dropDuplicates(["emp_id"])

df_dedup.show()

###7. Sort Data: Sort a DataFrame by date in descending order. 

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, "2024-01-10"),
    (2, "2024-03-15"),
    (3, "2023-12-01")
]

columns = ["id", "order_date"]

df = spark.createDataFrame(data, columns)

# Convert string to date (recommended if column is string)
from pyspark.sql.functions import to_date
df = df.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))

# Sort by date in descending order
df_sorted = df.orderBy(col("order_date").desc())

df_sorted.show()

###8. Cast Data Types: Convert a string column representing a date into a TimestampType. 

In [0]:
from pyspark.sql.functions import to_timestamp, col

# Sample DataFrame
data = [
    (1, "2024-01-10 14:30:00"),
    (2, "2024-03-15 09:45:10")
]

columns = ["id", "event_time"]

df = spark.createDataFrame(data, columns)

# Convert string column to TimestampType
df_casted = df.withColumn(
    "event_time",
    to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss")
)

df_casted.printSchema()
df_casted.show()

###9. Union DataFrames: Combine two DataFrames with the same schema using unionByName(). 

In [0]:
# First DataFrame
data1 = [
    (1, "Bharath", 28),
    (2, "Ravi", 30)
]

columns = ["id", "name", "age"]

df1 = spark.createDataFrame(data1, columns)

# Second DataFrame (same schema)
data2 = [
    (3, "Anu", 25),
    (4, "John", 32)
]

df2 = spark.createDataFrame(data2, columns)

# Union using column names
df_union = df1.unionByName(df2)

df_union.show()

###	10. Handle Nulls: Fill all null values in a column with a default value like 0 or Unknown. 


In [0]:
# Sample DataFrame
data = [
    (1, 5000, "Sales"),
    (2, None, "HR"),
    (3, 7000, None)
]

columns = ["emp_id", "salary", "department"]

df = spark.createDataFrame(data, columns)

# Fill null salary with 0
df_filled_salary = df.fillna({"salary": 0})

# Fill null department with "Unknown"
df_filled_all = df_filled_salary.fillna({"department": "Unknown"})

df_filled_all.show()

###11. Group By & Aggregate: Calculate the average salary per department. 

In [0]:
from pyspark.sql.functions import avg

# Sample DataFrame
data = [
    (1, "Sales", 5000),
    (2, "Sales", 7000),
    (3, "HR", 4000),
    (4, "HR", 6000),
    (5, "IT", 8000)
]

columns = ["emp_id", "department", "salary"]

df = spark.createDataFrame(data, columns)

# Group by department and calculate average salary
df_avg = df.groupBy("department") \
           .agg(avg("salary").alias("avg_salary"))

df_avg.show()

###12. Multiple Aggregations: Find the min, max, and count of a column in one groupBy operation. 

In [0]:
from pyspark.sql.functions import min, max, count

# Sample DataFrame
data = [
    (1, "Sales", 5000),
    (2, "Sales", 7000),
    (3, "HR", 4000),
    (4, "HR", 6000),
    (5, "IT", 8000)
]

columns = ["emp_id", "department", "salary"]

df = spark.createDataFrame(data, columns)

# Group by department and perform multiple aggregations
df_agg = df.groupBy("department").agg(
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    count("salary").alias("salary_count")
)

df_agg.show()

###13. Inner Join: Join two DataFrames on an employee_id column.

In [0]:
# Employees DataFrame
emp_data = [
    (1, "Bharath", "Sales"),
    (2, "Ravi", "HR"),
    (3, "Anu", "IT")
]

emp_columns = ["employee_id", "name", "department"]
df_emp = spark.createDataFrame(emp_data, emp_columns)

# Salary DataFrame
salary_data = [
    (1, 6000),
    (2, 5000),
    (4, 7000)   # No matching employee
]

salary_columns = ["employee_id", "salary"]
df_salary = spark.createDataFrame(salary_data, salary_columns)

# Inner Join on employee_id
df_joined = df_emp.join(
    df_salary,
    on="employee_id",
    how="inner"
)

df_joined.show()

###14. Left Join: Perform a left join and filter for records that didn't have a match in the right table. 

In [0]:
from pyspark.sql.functions import col

# Employees DataFrame (Left table)
emp_data = [
    (1, "Bharath"),
    (2, "Ravi"),
    (3, "Anu"),
    (4, "John")
]

emp_columns = ["employee_id", "name"]
df_emp = spark.createDataFrame(emp_data, emp_columns)

# Salary DataFrame (Right table)
salary_data = [
    (1, 6000),
    (2, 5000)
]

salary_columns = ["employee_id", "salary"]
df_salary = spark.createDataFrame(salary_data, salary_columns)

# Left Join
df_left = df_emp.join(
    df_salary,
    on="employee_id",
    how="left"
)

# Filter records that did NOT have a match in right table
df_no_match = df_left.filter(col("salary").isNull())

df_no_match.show()

###15. Pivot Table: Write code to pivot a "sales" column by "year". 

In [0]:
from pyspark.sql.functions import sum

# Sample DataFrame
data = [
    ("ProductA", 2022, 1000),
    ("ProductA", 2023, 1500),
    ("ProductB", 2022, 2000),
    ("ProductB", 2023, 2500)
]

columns = ["product", "year", "sales"]

df = spark.createDataFrame(data, columns)

# Pivot sales by year
df_pivot = df.groupBy("product") \
             .pivot("year") \
             .agg(sum("sales"))

df_pivot.show()

###16. Unpivot (Stack): Convert multiple columns (e.g., Q1, Q2, Q3 sales) into rows. 

In [0]:
from pyspark.sql.functions import expr

# Sample DataFrame
data = [
    ("ProductA", 1000, 1500, 2000),
    ("ProductB", 2000, 2500, 3000)
]

columns = ["product", "Q1", "Q2", "Q3"]

df = spark.createDataFrame(data, columns)

# Unpivot using stack()
df_unpivot = df.select(
    "product",
    expr("""
        stack(3, 
              'Q1', Q1,
              'Q2', Q2,
              'Q3', Q3
        ) as (quarter, sales)
    """)
)

df_unpivot.show()

###17. Self Join: Use a self-join to find pairs of employees working in the same department. 

In [0]:
from pyspark.sql.functions import col

# Sample DataFrame
data = [
    (1, "Bharath", "Sales"),
    (2, "Ravi", "Sales"),
    (3, "Anu", "HR"),
    (4, "John", "HR"),
    (5, "Mike", "IT")
]

columns = ["employee_id", "name", "department"]

df = spark.createDataFrame(data, columns)

# Create aliases for self join
df1 = df.alias("e1")
df2 = df.alias("e2")

# Self join on department and avoid duplicate/self pairing
df_pairs = df1.join(
    df2,
    (col("e1.department") == col("e2.department")) &
    (col("e1.employee_id") < col("e2.employee_id")),
    "inner"
).select(
    col("e1.employee_id").alias("emp1_id"),
    col("e1.name").alias("emp1_name"),
    col("e2.employee_id").alias("emp2_id"),
    col("e2.name").alias("emp2_name"),
    col("e1.department")
)

df_pairs.show()

###18. Join Optimization (Broadcast): Manually trigger a broadcast join for a small lookup table. 

- Use broadcast when one table is small (lookup/dimension table).
- Small table is sent to all executors.
- Avoids shuffle.
- Improves join performance for large-to-small joins.
- Spark automatically broadcasts small tables (based on spark.sql.autoBroadcastJoinThreshold), but broadcast() forces it manually.

In [0]:
from pyspark.sql.functions import broadcast

# Large DataFrame (Fact table)
fact_data = [
    (1, 101, 5000),
    (2, 102, 7000),
    (3, 103, 6000)
]

fact_columns = ["transaction_id", "dept_id", "amount"]
df_fact = spark.createDataFrame(fact_data, fact_columns)

# Small Lookup DataFrame (Dimension table)
lookup_data = [
    (101, "Sales"),
    (102, "HR"),
    (103, "IT")
]

lookup_columns = ["dept_id", "department"]
df_lookup = spark.createDataFrame(lookup_data, lookup_columns)

# Broadcast Join
df_joined = df_fact.join(
    broadcast(df_lookup),   # Force broadcast
    on="dept_id",
    how="inner"
)

df_joined.show()

###19. Anti Join: Find records in Table A that do not exist in Table B. 

In [0]:
# Table A
data_a = [
    (1, "Bharath"),
    (2, "Ravi"),
    (3, "Anu"),
    (4, "John")
]

columns_a = ["employee_id", "name"]
df_a = spark.createDataFrame(data_a, columns_a)

# Table B
data_b = [
    (1,),
    (3,)
]

columns_b = ["employee_id"]
df_b = spark.createDataFrame(data_b, columns_b)

# Left Anti Join
df_anti = df_a.join(
    df_b,
    on="employee_id",
    how="left_anti"
)

df_anti.show()

###20. Cross Join Warning: Write a cross-join but explain why it’s generally avoided. 

- Produces Cartesian product (rows_A × rows_B).
- Causes massive data explosion for large tables.
- Leads to heavy shuffle, memory pressure, and OOM errors.
- Can severely degrade cluster performance.
- Use only when Dataset is very small
- You intentionally need all possible combinations
- No join condition exists by design.

In [0]:
# DataFrame 1
data1 = [
    (1, "A"),
    (2, "B")
]

df1 = spark.createDataFrame(data1, ["id", "category"])

# DataFrame 2
data2 = [
    ("X",),
    ("Y",)
]

df2 = spark.createDataFrame(data2, ["type"])

# Enable cross join if disabled
spark.conf.set("spark.sql.crossJoin.enabled", "true")

# Cross Join
df_cross = df1.crossJoin(df2)

df_cross.show()

###21. Row Number: Assign a unique row number to each record within a department partition. 

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "HR", 4000),
    (3, "Charlie", "IT", 6000),
    (4, "David", "IT", 5500),
    (5, "Eve", "HR", 4500)
]

columns = ["emp_id", "emp_name", "department", "salary"]

df = spark.createDataFrame(data, columns)

window_spec = Window.partitionBy("department").orderBy(df.salary.desc())

df_with_rownum = df.withColumn(
    "row_number",
    row_number().over(window_spec)
)

df_with_rownum.show()

###22. Rank vs. Dense Rank: Write a code snippet showing the difference in rankings when ties exist. 

In [0]:
from pyspark.sql.functions import rank, dense_rank
from pyspark.sql.window import Window

data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "HR", 4000),
    (3, "Charlie", "HR", 4000),  # tie
    (4, "David", "HR", 3000)
]

columns = ["emp_id", "emp_name", "department", "salary"]

df = spark.createDataFrame(data, columns)

window_spec = Window.partitionBy("department").orderBy(df.salary.desc())

df_ranked = df.withColumn("rank", rank().over(window_spec)) \
              .withColumn("dense_rank", dense_rank().over(window_spec))

df_ranked.show()

###23. Lead/Lag: Find the previous day's sales for each store using the lag() function. 

In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window
from pyspark.sql.functions import col

data = [
    ("StoreA", "2024-01-01", 1000),
    ("StoreA", "2024-01-02", 1200),
    ("StoreA", "2024-01-03", 900),
    ("StoreB", "2024-01-01", 2000),
    ("StoreB", "2024-01-02", 1800)
]

columns = ["store", "date", "sales"]

df = spark.createDataFrame(data, columns)

window_spec = Window.partitionBy("store").orderBy("date")

df_with_lag = df.withColumn(
    "previous_day_sales",
    lag("sales", 1).over(window_spec)
)

df_with_lag.show()

###24. Cumulative Sum: Calculate a running total of transactions per user. 

In [0]:
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

data = [
    (1, "2024-01-01", 100),
    (1, "2024-01-02", 200),
    (1, "2024-01-03", 150),
    (2, "2024-01-01", 300),
    (2, "2024-01-02", 100)
]

columns = ["user_id", "transaction_date", "amount"]

df = spark.createDataFrame(data, columns)

###25. Moving Average: Calculate a 7-day rolling average for a stock price. 

In [0]:
from pyspark.sql.functions import avg
from pyspark.sql.window import Window

data = [
    ("2024-01-01", 100),
    ("2024-01-02", 105),
    ("2024-01-03", 110),
    ("2024-01-04", 120),
    ("2024-01-05", 115),
    ("2024-01-06", 125),
    ("2024-01-07", 130),
    ("2024-01-08", 140)
]

columns = ["trade_date", "stock_price"]

df = spark.createDataFrame(data, columns)

window_spec = Window.orderBy("trade_date") \
                    .rowsBetween(-6, 0)   # 6 previous rows + current row

df_moving_avg = df.withColumn(
    "7_day_moving_avg",
    avg("stock_price").over(window_spec)
)

df_moving_avg.show()

###26. Top N Per Group: Use window functions to find the top 3 highest-paid employees in each department. 

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "HR", 4000),
    (3, "Charlie", "HR", 4500),
    (4, "David", "HR", 3800),
    (5, "Eve", "IT", 7000),
    (6, "Frank", "IT", 6500),
    (7, "Grace", "IT", 7200),
    (8, "Helen", "IT", 6000)
]

columns = ["emp_id", "emp_name", "department", "salary"]

df = spark.createDataFrame(data, columns)

window_spec = Window.partitionBy("department") \
                    .orderBy(df.salary.desc())

df_top3 = df.withColumn(
    "row_num",
    row_number().over(window_spec)
).filter("row_num <= 3")

df_top3.show()

###27. Difference Between Dates: Calculate the number of days between two date columns using datediff(). 

In [0]:
from pyspark.sql.functions import datediff, to_date

data = [
    (1, "2024-01-01", "2024-01-10"),
    (2, "2024-02-05", "2024-02-20"),
    (3, "2024-03-15", "2024-03-18")
]

columns = ["id", "start_date", "end_date"]

df = spark.createDataFrame(data, columns)

df = df.withColumn("start_date", to_date("start_date")) \
       .withColumn("end_date", to_date("end_date"))

df_with_diff = df.withColumn(
    "days_difference",
    datediff("end_date", "start_date")
)

df_with_diff.show()

###28. Percent Rank: Calculate the percentile rank of students based on their marks. 

In [0]:
from pyspark.sql.functions import percent_rank
from pyspark.sql.window import Window

data = [
    (1, "Alice", 85),
    (2, "Bob", 90),
    (3, "Charlie", 75),
    (4, "David", 90),
    (5, "Eve", 60)
]

columns = ["student_id", "student_name", "marks"]

df = spark.createDataFrame(data, columns)

window_spec = Window.orderBy("marks")

df_percent_rank = df.withColumn(
    "percent_rank",
    percent_rank().over(window_spec)
)

df_percent_rank.show()

###29. Explode Arrays: Flatten a column containing a list of products into individual rows. 

In [0]:
from pyspark.sql.functions import explode

data = [
    (1, ["Laptop", "Mouse", "Keyboard"]),
    (2, ["Mobile", "Charger"]),
    (3, ["Tablet"])
]

columns = ["order_id", "products"]

df = spark.createDataFrame(data, columns)

df_exploded = df.withColumn(
    "product",
    explode("products")
).drop("products")

df_exploded.show()

###30. Flatten Structs: Access fields within a nested StructType and promote them to top-level columns. 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data = [
    (1, ("Alice", 30)),
    (2, ("Bob", 25))
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("user", StructType([
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True)
    ]))
])

df = spark.createDataFrame(data, schema)

df_flat = df.select(
    "id",
    "user.name",
    "user.age"
)

df_flat.show()

###31. JSON Parsing: Read a column containing JSON strings and parse them using a predefined schema. 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col

data = [
    (1, '{"name":"Alice","age":30}'),
    (2, '{"name":"Bob","age":25}')
]

columns = ["id", "json_string"]

df = spark.createDataFrame(data, columns)

json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df_parsed = df.withColumn(
    "parsed_json",
    from_json(col("json_string"), json_schema)
)

df_parsed.show(truncate=False)

df_final = df_parsed.select(
    "id",
    col("parsed_json.name").alias("name"),
    col("parsed_json.age").alias("age")
)

df_final.show()

###32. User Defined Functions (UDF): Register and apply a Python function to a DataFrame column. 

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

data = [
    (1, "alice"),
    (2, "bob"),
    (3, "charlie")
]

columns = ["id", "name"]

df = spark.createDataFrame(data, columns)

def to_uppercase(name):
    if name is not None:
        return name.upper()
    return None

uppercase_udf = udf(to_uppercase, StringType())

df_transformed = df.withColumn(
    "name_upper",
    uppercase_udf("name")
)

df_transformed.show()

###33. Conditional Logic: Use when() and otherwise() to create a status column based on age. 

In [0]:
from pyspark.sql.functions import when

data = [
    (1, "Alice", 15),
    (2, "Bob", 22),
    (3, "Charlie", 17),
    (4, "David", 30)
]

columns = ["id", "name", "age"]

df = spark.createDataFrame(data, columns)

df_status = df.withColumn(
    "status",
    when(df.age < 18, "Minor")
    .otherwise("Adult")
)

df_status.show()

###34. Substring Extraction: Extract the domain name from an email address column. 

In [0]:
from pyspark.sql.functions import split, substring_index

data = [
    (1, "alice@gmail.com"),
    (2, "bob@yahoo.com"),
    (3, "charlie@company.org")
]

columns = ["id", "email"]

df = spark.createDataFrame(data, columns)

df_domain = df.withColumn(
    "domain",
    split("email", "@")[1]
)

df_domain.show()

df_domain = df.withColumn(
    "domain",
    substring_index("email", "@", -1)
)

df_domain.show()

###35. MapPartitions: Demonstrate using mapPartitions to initialize an expensive connection (like a DB) once per partition.

- map() → Initializes connection for every row (expensive, inefficient).
- mapPartitions() → Initializes connection once per partition.
- Reduces overhead when interacting with:
  - Databases
  - External APIs
  - Machine learning models
  - Large file handles

In [0]:
data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eve")
]

columns = ["id", "name"]

df = spark.createDataFrame(data, columns)

def process_partition(partition):
    # Simulate expensive connection creation
    print("Opening DB connection")

    # Example: db_connection = create_connection()

    results = []
    for row in partition:
        # Simulate DB operation
        processed_value = row.name.upper()
        results.append((row.id, processed_value))

    # Simulate closing connection
    print("Closing DB connection")

    return iter(results)

rdd_result = df.rdd.mapPartitions(process_partition)

df_result = rdd_result.toDF(["id", "name_upper"])

df_result.show()

###36. Handling Corrupt Records: Read a file and use the _corrupt_record column to find bad rows. 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])

df = spark.read \
    .schema(schema) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("path/to/input.json")

df.filter(df._corrupt_record.isNotNull()).show(truncate=False)

df.filter(df._corrupt_record.isNull()).show()

###37. Missing Numbers in Sequence: Find gaps in a numeric sequence within a column. 

In [0]:
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window

data = [
    (1,),
    (2,),
    (3,),
    (6,),
    (7,),
    (10,)
]

columns = ["num"]

df = spark.createDataFrame(data, columns)

window_spec = Window.orderBy("num")

df_with_prev = df.withColumn(
    "prev_num",
    lag("num").over(window_spec)
)

df_gaps = df_with_prev.withColumn(
    "gap",
    col("num") - col("prev_num")
)

df_gaps.show()

df_missing_ranges = df_gaps.filter(col("gap") > 1) \
    .selectExpr(
        "prev_num + 1 as missing_start",
        "num - 1 as missing_end"
    )

df_missing_ranges.show()

###38. Repartition vs. Coalesce: Write code to reduce partitions from 100 to 1 and explain the choice. 

Coalesce(1)

- Used to reduce number of partitions.
- Avoids full shuffle.
- More efficient when decreasing partitions (100 → 1).
- Performs narrow transformation (moves data from fewer partitions).
- Best choice when:
    - Reducing partitions
    - Writing single output file
    - No need for data rebalancing

In [0]:
df_single_partition = df.coalesce(1)

repartition()

- Triggers full shuffle.
- Expensive operation.
- Used when:
    - Increasing partitions
    - Redistributing data evenly
    - Rebalancing skew

In [0]:
df_single_partition = df.repartition(1)

###39. Caching Strategy: Show how to cache() a DataFrame and then unpersist() it.

- Cache only if reused multiple times.
- Trigger an action immediately after caching.
- Always unpersist() when no longer needed.
- Avoid caching very large DataFrames unless necessary.

In [0]:
df_cached = df.cache()

# Trigger an action to materialize the cache
df_cached.count()

In [0]:
from pyspark import StorageLevel

df_persisted = df.persist(StorageLevel.MEMORY_AND_DISK)

df_persisted.count()

###40. Partitioned Writes: Write a DataFrame to Parquet format partitioned by year and month.

- partitionBy("year", "month") creates folder-based partitions.
- Improves query performance via partition pruning.
- Avoid very high-cardinality columns as partition columns.
- Use mode("overwrite") carefully in production.

In [0]:
from pyspark.sql.functions import year, month, col

df_with_partitions = df.withColumn("year", year(col("transaction_date"))) \
                       .withColumn("month", month(col("transaction_date")))

df_with_partitions.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("path/to/output/")

###41. Bucketing: Write code to save a table using bucketBy for optimized joins.

- Bucketing distributes data into a fixed number of files based on a hash of a column.
- Used mainly for improving join performance between large tables.
- Note: bucketBy() works only with saveAsTable() (Hive support enabled).

In [0]:
df.write \
    .format("parquet") \
    .bucketBy(8, "user_id") \
    .sortBy("user_id") \
    .mode("overwrite") \
    .saveAsTable("bucketed_users")

###42. Salting for Data Skew: Implement a "salting" technique to resolve a skewed join.  

- Salting Technique to Handle Data Skew in Joins
- One key (e.g., user_id = 1) appears millions of times → causes skew during join.
- Add a random "salt" to distribute skewed key across partitions.

In [0]:
from pyspark.sql.functions import col, floor, rand, concat_ws, lit, explode, array

data_large = [
    (1, "txn1"),
    (1, "txn2"),
    (1, "txn3"),
    (1, "txn4"),
    (2, "txn5"),
    (3, "txn6")
]

df_large = spark.createDataFrame(data_large, ["user_id", "transaction"])

In [0]:
data_small = [
    (1, "Premium"),
    (2, "Standard"),
    (3, "Basic")
]

df_small = spark.createDataFrame(data_small, ["user_id", "plan"])

Problem (Without Salting)

In [0]:
df_large.join(df_small, "user_id").show()

Step 1: Add Salt to Large (Skewed) Table

In [0]:
num_salts = 3

df_large_salted = df_large.withColumn(
    "salt",
    floor(rand() * num_salts)
).withColumn(
    "salted_key",
    concat_ws("_", col("user_id"), col("salt"))
)

Step 2: Expand Small Table Across Salts

In [0]:
salt_values = list(range(num_salts))

df_small_salted = df_small.withColumn(
    "salt",
    explode(array([lit(i) for i in salt_values]))
).withColumn(
    "salted_key",
    concat_ws("_", col("user_id"), col("salt"))
)

Step 3: Join on Salted Key

In [0]:
df_joined = df_large_salted.join(
    df_small_salted,
    on="salted_key",
    how="inner"
).drop("salt", "salted_key")

df_joined.show()

###43. Delta Upsert (Merge): Write a Delta Lake MERGE operation to update existing records and insert new ones. 

In [0]:
data_target = [
    (1, "Alice", 5000),
    (2, "Bob", 4000),
    (3, "Charlie", 3000)
]

df_target = spark.createDataFrame(data_target, ["id", "name", "salary"])

df_target.write.format("delta").mode("overwrite").saveAsTable("customers")

In [0]:
data_source = [
    (2, "Bob", 4500),        # Update
    (3, "Charlie", 3500),    # Update
    (4, "David", 6000)       # Insert
]

df_source = spark.createDataFrame(data_source, ["id", "name", "salary"])

df_source.createOrReplaceTempView("updates")

In [0]:
spark.sql("""
MERGE INTO customers AS target
USING updates AS source
ON target.id = source.id

WHEN MATCHED THEN
  UPDATE SET
    target.name = source.name,
    target.salary = source.salary

WHEN NOT MATCHED THEN
  INSERT (id, name, salary)
  VALUES (source.id, source.name, source.salary)
""")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "customers")

delta_table.alias("target") \
    .merge(
        df_source.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedUpdate(set={
        "name": "source.name",
        "salary": "source.salary"
    }) \
    .whenNotMatchedInsert(values={
        "id": "source.id",
        "name": "source.name",
        "salary": "source.salary"
    }) \
    .execute()

In [0]:
spark.table("customers").show()

###44. Delta Vacuum: Write the command to delete old file versions from a Delta table.

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "customers")

delta_table.vacuum(168)  # Retain 168 hours

### 45. Streaming Word Count: Write a basic Structured Streaming job to count words from a socket source. 

In [0]:
from pyspark.sql.functions import explode, split

df_stream = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

words = df_stream.select(
    explode(
        split(df_stream.value, " ")
    ).alias("word")
)

word_counts = words.groupBy("word").count()

query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

###46. Watermarking: Add a watermark to a streaming DataFrame to handle late-arriving data. 

In [0]:
from pyspark.sql.functions import window, col

df_stream = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

from pyspark.sql.functions import split, to_timestamp

parsed_df = df_stream.select(
    split(col("value"), ",").getItem(0).alias("user_id"),
    to_timestamp(split(col("value"), ",").getItem(1)).alias("event_time")
)

watermarked_df = parsed_df.withWatermark(
    "event_time",
    "10 minutes"
)

aggregated = watermarked_df.groupBy(
    window(col("event_time"), "5 minutes"),
    col("user_id")
).count()

query = aggregated.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

query.awaitTermination()

###47. Custom Schema Definition: Create a StructType schema manually rather than relying on inferSchema. 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

custom_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("join_date", DateType(), True)
])

df = spark.read \
    .format("csv") \
    .option("header", True) \
    .schema(custom_schema) \
    .load("path/to/input.csv")

###48. Checkpointing: Enable checkpointing for a streaming job or a long lineage RDD.  

In [0]:
from pyspark.sql.functions import explode, split

df_stream = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

words = df_stream.select(
    explode(split(df_stream.value, " ")).alias("word")
)

word_counts = words.groupBy("word").count()

query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("checkpointLocation", "path/to/checkpoint/dir") \
    .start()

query.awaitTermination()

###49. Identifying Data Skew: Write a query to count records per partition to detect skewness. 

In [0]:
from pyspark.sql.functions import spark_partition_id, count

df_partition_counts = df.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy(
    "partition_id"
).agg(
    count("*").alias("record_count")
).orderBy("partition_id")

df_partition_counts.show()


#+------------+------------+
#|partition_id|record_count|
#+------------+------------+
#|0           |500000      |
#|1           |120         |
#|2           |135         |
#|3           |140         |
#+------------+------------+

###50. Error Logging: Wrap a transformation in a try-except block and log the error.

In [0]:
import logging
from pyspark.sql.functions import col

# Configure logger
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger("spark_app")

try:
    # Example transformation (intentional error: wrong column name)
    df_transformed = df.withColumn(
        "salary_double",
        col("salry") * 2   # typo: 'salry' instead of 'salary'
    )

    df_transformed.show()

except Exception as e:
    logger.error(f"Error during transformation: {str(e)}", exc_info=True)

### 51. SCD Type 2 implementation

In [0]:
# ============================================================
# SCD Type 2 Implementation in PySpark / Databricks
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, sha2, concat_ws,
    coalesce, max as spark_max
)
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("SCD_Type2").getOrCreate()

# ============================================================
# 1. SETUP - Create Sample Tables
# ============================================================

# --- Existing DIM table (current state) ---
dim_data = [
    (1, "C001", "Alice",   "New York",   "Engineering", "2023-01-01", "9999-12-31", True),
    (2, "C002", "Bob",     "Chicago",    "Marketing",   "2023-01-01", "9999-12-31", True),
    (3, "C003", "Charlie", "Boston",     "Finance",     "2023-01-01", "9999-12-31", True),
]
dim_schema = StructType([
    StructField("surrogate_key",  IntegerType(),  False),
    StructField("customer_id",    StringType(),   False),
    StructField("name",           StringType(),   True),
    StructField("city",           StringType(),   True),
    StructField("department",     StringType(),   True),
    StructField("effective_date", StringType(),   True),
    StructField("end_date",       StringType(),   True),
    StructField("is_current",     BooleanType(),  True),
])
dim_df = spark.createDataFrame(dim_data, dim_schema)

# Write as Delta Table
dim_df.write.format("delta").mode("overwrite").saveAsTable("dim_customers")
print("✅ Initial DIM table created:")
spark.table("dim_customers").show()


# --- Incoming staging data (new/changed records) ---
stg_data = [
    ("C001", "Alice",   "San Francisco", "Engineering"),  # city changed
    ("C002", "Bob",     "Chicago",       "Marketing"),    # no change
    ("C004", "Diana",   "Seattle",       "HR"),           # new customer
]
stg_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("name",        StringType(), True),
    StructField("city",        StringType(), True),
    StructField("department",  StringType(), True),
])
stg_df = spark.createDataFrame(stg_data, stg_schema)
print("📦 Staging (incoming) data:")
stg_df.show()


# ============================================================
# 2. SCD TYPE 2 - CORE LOGIC
# ============================================================

from datetime import date

today = str(date.today())           # effective date for new rows
max_date = "9999-12-31"            # open-ended "current" marker

# Step 1: Load current DIM records
dim_current = spark.table("dim_customers").filter(col("is_current") == True)

# Step 2: Join staging with current dimension
joined = stg_df.alias("stg").join(
    dim_current.alias("dim"),
    on="customer_id",
    how="left"
)

# Step 3: Identify record types
#   - NEW    : no match in dim (dim.customer_id IS NULL)
#   - CHANGED: match but at least one tracked column differs
#   - SAME   : match and nothing changed

SCD_COLS = ["city", "department"]   # columns we track for changes

# Build a change-detection flag
change_condition = " OR ".join(
    [f"stg.{c} <> dim.{c}" for c in SCD_COLS]
)

records_new = joined.filter(col("dim.customer_id").isNull())
records_changed = joined.filter(
    col("dim.customer_id").isNotNull() &
    (
        (col("stg.city")       != col("dim.city")) |
        (col("stg.department") != col("dim.department"))
    )
)
records_same = joined.filter(
    col("dim.customer_id").isNotNull() &
    (col("stg.city")       == col("dim.city")) &
    (col("stg.department") == col("dim.department"))
)

print(f"🆕 New records   : {records_new.count()}")
print(f"✏️  Changed records: {records_changed.count()}")
print(f"✔️  Unchanged     : {records_same.count()}")


# ============================================================
# 3. APPLY CHANGES USING DELTA MERGE
# ============================================================

delta_table = DeltaTable.forName(spark, "dim_customers")

# --- Step A: Expire old rows for changed records ---
#   Set is_current=False and end_date=today for changed records
changed_keys = records_changed.select("stg.customer_id")

delta_table.alias("dim").merge(
    changed_keys.alias("upd"),
    "dim.customer_id = upd.customer_id AND dim.is_current = true"
).whenMatchedUpdate(set={
    "is_current": lit(False),
    "end_date":   lit(today)
}).execute()

print("✅ Expired old rows for changed customers:")
spark.table("dim_customers").show()


# --- Step B: Insert new rows (new + changed customers) ---

# Get the max surrogate key to continue sequencing
max_sk = spark.table("dim_customers").agg(
    spark_max("surrogate_key")
).collect()[0][0]

# Build insert DataFrame: new customers + new version of changed customers
new_rows_new = records_new.select(
    col("stg.customer_id"),
    col("stg.name"),
    col("stg.city"),
    col("stg.department"),
)
new_rows_changed = records_changed.select(
    col("stg.customer_id"),
    col("stg.name"),
    col("stg.city"),
    col("stg.department"),
)

rows_to_insert = new_rows_new.union(new_rows_changed)

# Add surrogate key using monotonically increasing id + offset
from pyspark.sql.functions import monotonically_increasing_id

rows_to_insert = (
    rows_to_insert
    .withColumn("_rn", monotonically_increasing_id())
    .withColumn("surrogate_key", (col("_rn") + lit(max_sk + 1)).cast("int"))
    .withColumn("effective_date", lit(today))
    .withColumn("end_date",       lit(max_date))
    .withColumn("is_current",     lit(True))
    .drop("_rn")
)

rows_to_insert.write.format("delta").mode("append").saveAsTable("dim_customers")

print("\n🎉 Final DIM table after SCD Type 2 merge:")
spark.table("dim_customers").orderBy("customer_id", "effective_date").show()


# ============================================================
# 4. HELPER: REUSABLE SCD TYPE 2 FUNCTION
# ============================================================

def apply_scd2(
    spark,
    target_table: str,
    source_df,
    business_key: str,
    tracked_cols: list,
    surrogate_key_col: str = "surrogate_key",
    effective_col: str = "effective_date",
    end_col: str = "end_date",
    current_flag_col: str = "is_current",
):
    """
    Generic SCD Type 2 implementation using Delta Lake.

    Args:
        spark          : SparkSession
        target_table   : Delta table name (string)
        source_df      : Incoming staging DataFrame
        business_key   : Business key column name (e.g. 'customer_id')
        tracked_cols   : List of columns to track for changes
        surrogate_key_col : Name of the surrogate key column
        effective_col  : Effective date column
        end_col        : End date column
        current_flag_col  : Boolean current-row flag column
    """
    from delta.tables import DeltaTable
    from datetime import date

    today    = str(date.today())
    max_date = "9999-12-31"

    dim_current = spark.table(target_table).filter(col(current_flag_col) == True)

    joined = source_df.alias("stg").join(
        dim_current.alias("dim"), on=business_key, how="left"
    )

    # Detect changes
    change_filter = None
    for c in tracked_cols:
        cond = col(f"stg.{c}") != col(f"dim.{c}")
        change_filter = cond if change_filter is None else (change_filter | cond)

    records_new     = joined.filter(col(f"dim.{business_key}").isNull())
    records_changed = joined.filter(col(f"dim.{business_key}").isNotNull() & change_filter)

    # Expire old rows
    delta_table = DeltaTable.forName(spark, target_table)
    changed_keys = records_changed.select(f"stg.{business_key}")

    delta_table.alias("dim").merge(
        changed_keys.alias("upd"),
        f"dim.{business_key} = upd.{business_key} AND dim.{current_flag_col} = true"
    ).whenMatchedUpdate(set={
        current_flag_col: lit(False),
        end_col:          lit(today)
    }).execute()

    # Insert new versions
    cols_to_select = [business_key] + [
        c for c in source_df.columns if c != business_key
    ]
    insert_df = (
        records_new.select([f"stg.{c}" for c in cols_to_select])
        .union(records_changed.select([f"stg.{c}" for c in cols_to_select]))
    )

    max_sk = spark.table(target_table).agg(
        spark_max(surrogate_key_col)
    ).collect()[0][0] or 0

    insert_df = (
        insert_df
        .withColumn("_rn",           monotonically_increasing_id())
        .withColumn(surrogate_key_col, (col("_rn") + lit(max_sk + 1)).cast("int"))
        .withColumn(effective_col,     lit(today))
        .withColumn(end_col,           lit(max_date))
        .withColumn(current_flag_col,  lit(True))
        .drop("_rn")
    )

    insert_df.write.format("delta").mode("append").saveAsTable(target_table)
    print(f"✅ SCD2 applied to '{target_table}': "
          f"{records_new.count()} new, {records_changed.count()} changed.")


# ============================================================
# 5. QUERY PATTERNS
# ============================================================

print("\n--- Current snapshot (all active records) ---")
spark.sql("""
    SELECT * FROM dim_customers WHERE is_current = true
""").show()

print("\n--- Full history for customer C001 ---")
spark.sql("""
    SELECT customer_id, name, city, department, effective_date, end_date, is_current
    FROM dim_customers
    WHERE customer_id = 'C001'
    ORDER BY effective_date
""").show()

print("\n--- Point-in-time lookup (as of 2023-06-01) ---")
spark.sql("""
    SELECT * FROM dim_customers
    WHERE customer_id = 'C001'
      AND effective_date <= '2023-06-01'
      AND end_date       >= '2023-06-01'
""").show()